# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""

    WITH bounds AS (

        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}

    ),

    windowed AS (

        SELECT f.client_hash_id, f.content_hash_id,

               SUM(CASE
                   WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                   THEN f.gsc_impressions ELSE 0
               END) AS imp_last30,

               SUM(CASE
                   WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                   THEN f.gsc_impressions ELSE 0
               END) AS imp_prev30,

               SUM(CASE
                   WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                   THEN f.gsc_clicks ELSE 0
               END) AS clk_last30,

               AVG(CASE
                   WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                   THEN f.gsc_avg_position
               END) AS pos_last30,

               AVG(CASE
                   WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                   THEN CASE
                       WHEN EXTRACT(DOW FROM f.report_date) IN (0, 6)
                       THEN 1.0 ELSE 0.0
                   END
               END) AS weekend_share

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 100

    )

    SELECT * FROM windowed

""").df()

print(f'{len(features):,} content items with enough history')

features.head()

## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

data['is_declining'] = (
    data['imp_last30'] < 0.8 * data['imp_prev30']
).astype(int)

feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share'
]

model_data = data.dropna(
    subset=feature_cols + ['client_hash_id']
)

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

# Split by CLIENT, not by individual rows
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]
y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
).fit(X_tr, y_tr)

print(
    f'train clients: {groups.iloc[train_idx].nunique():,}'
)

print(
    f'test clients: {groups.iloc[test_idx].nunique():,}'
)

print(
    f'base rate (always predict majority): '
    f'{max(y_te.mean(), 1 - y_te.mean()):.3f}'
)

print(
    classification_report(
        y_te,
        model.predict(X_te),
        digits=3
    )
)


In [ ]:
print(data.columns.tolist())

In [ ]:
feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'kept_impressions'
]

model_data = data.dropna(
    subset=feature_cols + ['client_hash_id']
)

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]
y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
).fit(X_tr, y_tr)

print(f'train clients: {groups.iloc[train_idx].nunique():,}')
print(f'test clients: {groups.iloc[test_idx].nunique():,}')

print(
    f'base rate (always predict majority): '
    f'{max(y_te.mean(), 1 - y_te.mean()):.3f}'
)

print(
    classification_report(
        y_te,
        model.predict(X_te),
        digits=3
    )
)

In [ ]:
# Create a simple, interpretable opportunity score

results = data.copy()

results['decline_rate'] = (
    1 - results['imp_last30'] / results['imp_prev30'].replace(0, np.nan)
)

results['decline_rate'] = results['decline_rate'].clip(lower=0, upper=1)

# Higher score = stronger reason to investigate
results['opportunity_score'] = (
    0.6 * results['decline_rate']
    + 0.4 * (
        1 - results['kept_impressions'] /
        results['kept_impressions'].max()
    )
)

# Reason code
results['reason'] = np.select(
    [
        results['decline_rate'] >= 0.5,
        results['kept_impressions'] <
        results['kept_impressions'].median()
    ],
    [
        'Strong traffic decline',
        'Low kept impressions'
    ],
    default='Monitor'
)

ranked = (
    results[
        [
            'client_hash_id',
            'content_hash_id',
            'imp_last30',
            'imp_prev30',
            'kept_impressions',
            'opportunity_score',
            'reason'
        ]
    ]
    .dropna(subset=['opportunity_score'])
    .sort_values('opportunity_score', ascending=False)
    .head(20)
)

ranked

In [ ]:
# Build the full ranked queue for ML-07.
# Do not export client/content hashes.

results["reason_code"] = np.select(
    [
        results["decline_rate"] >= 0.5,
        results["kept_impressions"] < results["kept_impressions"].median(),
    ],
    [
        "strong_traffic_decline",
        "low_kept_impressions",
    ],
    default="monitor",
)

results["action"] = np.select(
    [
        results["imp_last30"].fillna(0) == 0,
        results["decline_rate"] >= 0.5,
        results["decline_rate"] >= 0.2,
    ],
    [
        "investigate_traffic_loss",
        "review_or_refresh",
        "monitor_declining_performance",
    ],
    default="maintain_or_monitor",
)

queue = (
    results[
        [
            "imp_prev30",
            "imp_last30",
            "kept_impressions",
            "decline_rate",
            "opportunity_score",
            "reason_code",
            "action",
        ]
    ]
    .dropna(subset=["opportunity_score"])
    .sort_values("opportunity_score", ascending=False)
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

print("Total ranked items:", f"{len(queue):,}")
display(queue.head(20))

In [ ]:
print("Action distribution:")
display(
    queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="content_items")
)

print("\nReason-code distribution:")
display(
    queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="content_items")
)

print("\nTop 20 score range:")
print(queue.head(20)["opportunity_score"].agg(["min", "max"]))

In [ ]:
print("Decline rate:")
display(results["decline_rate"].describe())

print("\nKept impressions:")
display(results["kept_impressions"].describe())

print("\nOpportunity score:")
display(results["opportunity_score"].describe())

print("\nUnique scores in the top 20:")
print(queue.head(20)["opportunity_score"].nunique())

In [ ]:
audit_columns = [
    c for c in [
        "imp_prev30",
        "imp_last30",
        "clk_last30",
        "pos_last30",
        "visible_queries",
        "rare_share",
        "anon_share",
        "kept_impressions",
    ]
    if c in data.columns
]

summary = data[audit_columns].describe().T.round(2)
display(summary)


In [ ]:
import numpy as np
import pandas as pd

audit = data.copy()

audit["is_declining"] = (
    audit["imp_last30"] < 0.8 * audit["imp_prev30"]
)

# Signal 1: previous-window visibility
audit["visibility_group"] = np.where(
    audit["imp_prev30"] >= 500,
    "500_plus",
    "under_500"
)

print("Signal 1 — Previous-window visibility")
display(
    audit.groupby("visibility_group", observed=False)
    .agg(
        items=("is_declining", "size"),
        decline_rate=("is_declining", "mean"),
        median_previous_impressions=("imp_prev30", "median"),
    )
)

# Signal 2: zero versus nonzero latest impressions
audit["latest_group"] = np.where(
    audit["imp_last30"] == 0,
    "zero_latest_impressions",
    "nonzero_latest_impressions"
)

print("\nSignal 2 — Latest impressions")
display(
    audit.groupby("latest_group", observed=False)
    .agg(
        items=("is_declining", "size"),
        decline_rate=("is_declining", "mean"),
        median_previous_impressions=("imp_prev30", "median"),
    )
)

# Signal 3: position groups
position_audit = audit[audit["pos_last30"].notna()].copy()
position_audit["position_group"] = pd.cut(
    position_audit["pos_last30"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["1–3", "4–10", "11–20", "Over 20"]
)

print("\nSignal 3 — Position groups")
display(
    position_audit.groupby("position_group", observed=False)
    .agg(
        items=("is_declining", "size"),
        decline_rate=("is_declining", "mean"),
        median_latest_impressions=("imp_last30", "median"),
    )
)

In [ ]:
# Inspect the highest-scoring items with a minimum historical-volume check.
results["volume_eligible"] = results["imp_prev30"] >= 500

print("Items with at least 500 previous-window impressions:")
print(results["volume_eligible"].value_counts())

print("\nTop 20 among volume-eligible items:")
eligible_queue = (
    results[results["volume_eligible"]]
    .sort_values("opportunity_score", ascending=False)
    .head(20)
)

display(
    eligible_queue[
        [
            "imp_prev30",
            "imp_last30",
            "kept_impressions",
            "decline_rate",
            "opportunity_score",
            "reason",
        ]
    ]
)

print("\nUnique scores in this top 20:")
print(eligible_queue["opportunity_score"].nunique())

In [ ]:
results['action'] = np.select(
    [
        results['imp_last30'] == 0,
        results['decline_rate'] >= 0.5,
        results['decline_rate'] >= 0.2
    ],
    [
        'Investigate traffic loss',
        'Review or refresh content',
        'Monitor declining performance'
    ],
    default='Maintain / monitor'
)

action_summary = (
    results['action']
    .value_counts()
    .rename_axis('recommended_action')
    .reset_index(name='content_items')
)

action_summary

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'validation': [
        'Random row split',
        'Client-held-out split',
        'Client-held-out + kept_impressions'
    ],
    'accuracy': [
        0.656,
        0.611,
        0.663
    ],
    'class_1_f1': [
        0.756,
        0.704,
        0.745
    ],
    'baseline_accuracy': [
        0.633,
        0.677,
        0.677
    ]
})

comparison

In [6]:
# Lightweight notebook smoke check — no dataset query
print("Full-release notebook opened successfully.")
print("DuckDB workflow: configured for Hugging Face Parquet data.")
print("Heavy data queries have not been verified by this check.")

Full-release notebook opened successfully.
DuckDB workflow: configured for Hugging Face Parquet data.
Heavy data queries have not been verified by this check.


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
